# Historical Extrapolation of Survey Expectations

This notebook:
1. Loads best-validated models for all series from `model_final.pkl`
2. Rebuilds the shared Mahalanobis support checker from combined calibration waves
3. Extrapolates fitted expectations backward through the full news corpus
4. Validates the extrapolated series against known historical episodes
5. Saves one CSV per series for downstream use

**Four series:**
- `earnings_growth`  — Ridge
- `dividend_growth`  — Ridge
- `returns_1yr`      — Ridge (S&P 500 1-year expected return)
- `returns_10yr`     — Ridge (S&P 500 10-year expected return)

**Requires:** `news_df`, `article_embeddings`, `p_neutral_all`, `shiller_df`,
`all_results`, `SHILLER_FEATURES`, `get_shiller_features_for_wave`,
`aggregate_to_waves` from the main pipeline in memory.

## 1. Configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pickle, warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf

SEED = 42
np.random.seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("./output")
CACHE_DIR  = Path("/hpc/dctrl/ah620/storgae")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# ── Model choice per series ───────────────────────────────────────────────────
# All Ridge — RF boundary problem makes pre-survey extrapolation flat/uninformative
MODEL_CHOICE = {
    "earnings_growth": "ridge",
    "dividend_growth": "ridge",
    "returns_1yr":     "ridge",
    "returns_10yr":    "ridge",
}

# ── Ridge alpha grid ──────────────────────────────────────────────────────────
RIDGE_ALPHAS = np.logspace(-3, 6, 50)

# ── Extrapolation aggregation window ─────────────────────────────────────────
AGG_WINDOW_DAYS       = 30
RECENCY_WEIGHTING     = "exponential"
RECENCY_HALFLIFE_DAYS = 7

# ── Mahalanobis support check ─────────────────────────────────────────────────
PCA_COMPONENTS_MAHAL = 30
MAHAL_PERCENTILE     = 0.99

# ── NBER recession dates ──────────────────────────────────────────────────────
NBER_RECESSIONS = [
    ("1960-04", "1961-02"), ("1969-12", "1970-11"), ("1973-11", "1975-03"),
    ("1980-01", "1980-07"), ("1981-07", "1982-11"), ("1990-07", "1991-03"),
    ("2001-03", "2001-11"), ("2007-12", "2009-06"), ("2020-02", "2020-04"),
]

N_MACRO = len(SHILLER_FEATURES)

# ── Returns series have limited history (2000Q2 onwards) ─────────────────────
# Pre-survey extrapolation for returns only goes back to the news corpus start
# since there is no ground truth to calibrate against before 2000.
# The plot will clearly show where the survey sample begins.
RETURNS_SERIES = {"returns_1yr", "returns_10yr"}

print("Configuration loaded.")
print(f"Series to extrapolate: {list(MODEL_CHOICE.keys())}")

## 2. Fit best model on full calibration sample and save

Each model is refit on **all** survey waves (not just the tuning window)
at the hyperparameters validated by the walk-forward evaluation.
This is the model used for extrapolation.

In [ ]:
def fit_ridge_full(Z, y, alphas=RIDGE_ALPHAS):
    sc = StandardScaler().fit(Z)
    Zs = sc.transform(Z)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rc = RidgeCV(alphas=alphas, gcv_mode="auto",
                     scoring="neg_mean_squared_error")
        rc.fit(Zs, y)
    ridge = Ridge(alpha=rc.alpha_).fit(Zs, y)
    return ridge, sc, rc.alpha_


def build_mahal_checker(Z):
    n_mah   = min(PCA_COMPONENTS_MAHAL, len(Z) - 1)
    pca     = PCA(n_components=n_mah, random_state=SEED).fit(Z)
    Z_red   = pca.transform(Z)
    lw      = LedoitWolf().fit(Z_red)
    mu      = Z_red.mean(axis=0)
    inv_cov = np.linalg.pinv(lw.covariance_)
    thresh  = np.quantile(
        np.sqrt(np.einsum("ij,jk,ik->i", Z_red-mu, inv_cov, Z_red-mu)),
        MAHAL_PERCENTILE
    )
    return pca, mu, inv_cov, thresh


# ── Build shared Mahalanobis checker from ALL series combined ─────────────────
print("Building shared Mahalanobis checker (all calibration waves)...")
Z_combined = np.vstack([res["Z"] for res in all_results.values()])
print(f"  Combined matrix: {Z_combined.shape}")
pca_shared, mu_shared, inv_cov_shared, thresh_shared = build_mahal_checker(Z_combined)
print(f"  Shared threshold ({MAHAL_PERCENTILE:.0%}): {thresh_shared:.4f}")


# ── Fit Ridge on full calibration sample for each series ─────────────────────
saved_models = {}

for series_name, res in all_results.items():
    if series_name not in MODEL_CHOICE:
        print(f"Skipping {series_name} (not in MODEL_CHOICE)")
        continue

    Z        = res["Z"]
    y        = res["y"]
    waves_df = res["waves_df"]
    bp       = res["best_params"]

    print(f"\n{'='*55}")
    print(f"  {series_name}  →  RIDGE")
    print(f"{'='*55}")

    model, scaler, alpha = fit_ridge_full(Z, y)
    y_insample  = model.predict(scaler.transform(Z))
    r2_full     = 1 - np.sum((y - y_insample)**2) / np.sum((y - y.mean())**2)
    print(f"  Ridge α={alpha:.2e}  full-sample R²={r2_full:.3f}")

    model_obj = {
        "type":             "ridge",
        "model":            model,
        "scaler":           scaler,
        "alpha":            alpha,
        "r2_full":          r2_full,
        # Shared support checker (same for all series)
        "pca_mah":          pca_shared,
        "mu_mah":           mu_shared,
        "inv_cov_mah":      inv_cov_shared,
        "mahal_thresh":     thresh_shared,
        # Metadata
        "best_params":      bp,
        "shiller_features": SHILLER_FEATURES,
        "n_macro":          N_MACRO,
        "neutral_threshold": bp["neutral_threshold"],
        "series":           series_name,
        "waves_df":         waves_df,
        "y_calibration":    y,
        "Z_calibration":    Z,
        "wf_r2":            res["wf_r2"],
        "wf_rmse":          res["wf_rmse"],
    }

    # Save model_final.pkl
    save_path = OUTPUT_DIR / series_name / "model_final.pkl"
    (OUTPUT_DIR / series_name).mkdir(exist_ok=True, parents=True)
    with open(save_path, "wb") as f:
        pickle.dump(model_obj, f)
    print(f"  Saved → {save_path}")
    saved_models[series_name] = model_obj

print(f"\nAll {len(saved_models)} models saved with shared Mahal threshold={thresh_shared:.4f}")

## 3. Build historical wave grid

Generate quarterly wave dates spanning the full news corpus.
For each date, we aggregate the preceding 30-day window of headlines
using the same pipeline as calibration — recency weighting, sentiment
filtering, and Shiller macro feature lookup.

In [ ]:
def recency_weights_extrap(article_dates, wave_date, halflife_days=7):
    days_back = (wave_date - article_dates).dt.days.values
    return 0.5 ** (days_back / halflife_days)


def aggregate_wave(wave_date, news_df, article_embeddings,
                   p_neutral_all, neutral_threshold,
                   shiller_df, window_days=AGG_WINDOW_DAYS):
    """
    Aggregate one wave date → feature vector (news + macro).
    Returns (z_wave, n_raw, n_kept) or None if no articles.
    """
    mask = (
        (news_df["date"] >= wave_date - pd.Timedelta(days=window_days)) &
        (news_df["date"] <  wave_date)
    )
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return None, 0, 0

    emb  = article_embeddings[idx]
    good = ~np.isnan(emb).any(axis=1)
    emb, idx = emb[good], idx[good]
    if len(idx) == 0:
        return None, len(idx), 0

    n_raw = len(idx)

    # Sentiment filter
    if p_neutral_all is not None and neutral_threshold < 1.0:
        keep = p_neutral_all[idx] <= neutral_threshold
        if keep.sum() == 0:
            keep[np.argmin(p_neutral_all[idx])] = True
        emb, idx = emb[keep], idx[keep]
    n_kept = len(idx)

    # Recency-weighted mean
    w      = recency_weights_extrap(news_df.iloc[idx]["date"], wave_date)
    w_norm = w / w.sum()
    z_news = (emb * w_norm[:, None]).sum(axis=0)

    # Shiller macro features
    macro = get_shiller_features_for_wave(wave_date, shiller_df)
    z     = np.concatenate([z_news, macro])

    return z, n_raw, n_kept


# ── Generate quarterly wave dates for full corpus span ────────────────────────
corpus_start = news_df["date"].min()
corpus_end   = news_df["date"].max()

# Quarterly dates at end of each quarter
wave_dates_all = pd.date_range(
    start=corpus_start + pd.Timedelta(days=AGG_WINDOW_DAYS),
    end=corpus_end,
    freq="QE"   # quarter-end
)

print(f"Corpus span: {corpus_start.date()} → {corpus_end.date()}")
print(f"Generated {len(wave_dates_all):,} quarterly wave dates")
print(f"  First: {wave_dates_all[0].date()}")
print(f"  Last:  {wave_dates_all[-1].date()}")

## 4. Extrapolate — predict for every quarterly wave

For each quarterly wave date, aggregate the news embeddings + Shiller
features, compute Mahalanobis distance from the calibration distribution,
and generate a prediction from the best model.

In [ ]:
from tqdm import tqdm

extrap_results = {}

for series_name, model_obj in saved_models.items():
    print(f"\n{'='*55}")
    print(f"  Extrapolating: {series_name}")
    print(f"{'='*55}")

    choice    = model_obj["type"]
    neutral_t = model_obj["neutral_threshold"]
    pca_mah   = model_obj["pca_mah"]
    mu_mah    = model_obj["mu_mah"]
    inv_cov   = model_obj["inv_cov_mah"]
    thresh    = model_obj["mahal_thresh"]

    def mahal_dist(z):
        z_red = pca_mah.transform(z.reshape(1, -1))
        diff  = z_red - mu_mah
        return float(np.sqrt(diff @ inv_cov @ diff.T))

    def predict_wave(z):
        if choice == "ridge":
            return float(model_obj["model"].predict(
                model_obj["scaler"].transform(z.reshape(1, -1))
            )[0])
        else:
            return float(model_obj["model"].predict(z.reshape(1, -1))[0])

    records = []
    n_no_coverage = 0

    for wd in tqdm(wave_dates_all, desc=series_name):
        z, n_raw, n_kept = aggregate_wave(
            wd, news_df, article_embeddings,
            p_neutral_all, neutral_t, shiller_df,
        )
        if z is None:
            n_no_coverage += 1
            continue

        # Impute NaN macro features with calibration means
        macro_start = -model_obj["n_macro"]
        col_means   = np.nanmean(model_obj["Z_calibration"][:, macro_start:], axis=0)
        nan_mask    = np.isnan(z[macro_start:])
        z[macro_start:][nan_mask] = col_means[nan_mask]

        pred = predict_wave(z)
        d    = mahal_dist(z)

        records.append({
            "wave_date":       wd,
            "predicted":       pred,
            "mahal_dist":      d,
            "out_of_support":  d > thresh,
            "n_articles_raw":  n_raw,
            "n_articles_kept": n_kept,
            "in_calibration":  False,  # will be updated below
        })

    extrap_df = pd.DataFrame(records)

    # Mark waves that overlap with the calibration survey sample
    cal_dates = set(model_obj["waves_df"]["wave_date"].dt.to_period("Q"))
    extrap_df["in_calibration"] = extrap_df["wave_date"].dt.to_period("Q").isin(cal_dates)

    # Merge calibration realised values where available
    cal_map = dict(zip(
        model_obj["waves_df"]["wave_date"].dt.to_period("Q"),
        model_obj["y_calibration"]
    ))
    extrap_df["realized"] = extrap_df["wave_date"].dt.to_period("Q").map(cal_map)

    extrap_results[series_name] = extrap_df

    print(f"  Extrapolated: {len(extrap_df):,} waves")
    print(f"  No coverage:  {n_no_coverage:,} waves skipped")
    print(f"  Out-of-support (Mahal > thresh): "
          f"{extrap_df['out_of_support'].sum():,} "
          f"({extrap_df['out_of_support'].mean():.1%})")
    print(f"  Calibration overlap: {extrap_df['in_calibration'].sum():,} waves")

In [ ]:
# ── Save extrapolated series to CSV ──────────────────────────────────────────
for series_name, df in extrap_results.items():
    save_path = OUTPUT_DIR / f"extrapolated_{series_name}.csv"
    df.to_csv(save_path, index=False)
    print(f"Saved {series_name}: {len(df):,} rows → {save_path}")

print("\nColumn descriptions:")
print("  wave_date       — quarter-end date")
print("  predicted       — model prediction (survey expectation units)")
print("  realized        — actual survey value (NaN for pre-survey period)")
print("  mahal_dist      — Mahalanobis distance from calibration distribution")
print("  out_of_support  — True if wave is outside calibration support")
print("  in_calibration  — True if wave overlaps with survey sample")
print("  n_articles_*    — article counts before/after sentiment filter")

## 5. Full series plot with recession shading

Plot the extrapolated series alongside the realized survey values.
NBER recession periods are shaded. Out-of-support waves are marked.

In [ ]:
fig, axes = plt.subplots(len(extrap_results), 1,
                          figsize=(16, 6 * len(extrap_results)))
if len(extrap_results) == 1:
    axes = [axes]

for ax, (series_name, df) in zip(axes, extrap_results.items()):
    choice = MODEL_CHOICE[series_name]

    # ── Recession shading ────────────────────────────────────────────────────
    for start, end in NBER_RECESSIONS:
        s = pd.Timestamp(start + "-01")
        e = pd.Timestamp(end   + "-01")
        if e >= df["wave_date"].min() and s <= df["wave_date"].max():
            ax.axvspan(s, e, alpha=0.12, color="gray", zorder=0)

    # ── Out-of-support shading ───────────────────────────────────────────────
    oos = df[df["out_of_support"]]
    for _, row in oos.iterrows():
        ax.axvspan(row["wave_date"] - pd.Timedelta(days=45),
                   row["wave_date"] + pd.Timedelta(days=45),
                   alpha=0.15, color="#e07b39", zorder=0)

    # ── Predicted series ─────────────────────────────────────────────────────
    # Pre-calibration
    pre  = df[~df["in_calibration"]]
    cal  = df[ df["in_calibration"]]

    ax.plot(pre["wave_date"], pre["predicted"], "-",
            color="#2c5f8a", lw=1.5, alpha=0.85,
            label=f"Predicted ({choice.upper()}) — pre-survey")
    ax.plot(cal["wave_date"], cal["predicted"], "-",
            color="#2c5f8a", lw=1.5, alpha=0.4,
            label="Predicted — calibration period")

    # Realised survey
    realized = df.dropna(subset=["realized"])
    ax.plot(realized["wave_date"], realized["realized"],
            "o-", color="#c0392b", lw=2, ms=3,
            label="Realized survey")

    # Calibration boundary
    first_cal = df[df["in_calibration"]]["wave_date"].min()
    ax.axvline(first_cal, color="black", ls="--", lw=1, alpha=0.5,
               label="Survey sample start")

    ax.set_title(
        f"{series_name.replace('_',' ').title()}  "
        f"[{choice.upper()} model — WF R²={all_results[series_name]['wf_r2']:.3f}]",
        fontweight="bold", fontsize=11
    )
    ax.set_xlabel("Wave date"); ax.set_ylabel("Expected growth")
    ax.grid(alpha=0.25)

    # Legend with recession note
    handles, labels = ax.get_legend_handles_labels()
    handles.append(mpatches.Patch(color="gray",  alpha=0.3, label="NBER recession"))
    handles.append(mpatches.Patch(color="#e07b39", alpha=0.3, label="Out of support"))
    ax.legend(handles=handles, fontsize=8, loc="upper left")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "extrapolated_series.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR / 'extrapolated_series.png'}")

## 6. Validation against known historical episodes

Check whether the extrapolated series moves in the right direction during
well-documented periods of high and low earnings/dividend expectations.
This is the key qualitative check before using the series downstream.

In [ ]:
VALIDATION_EPISODES = {
    "earnings_growth": [
        ("1982-07", "1983-06", "post-recession recovery",    "high"),
        ("1990-07", "1991-06", "Gulf War recession",         "low"),
        ("1999-01", "2001-06", "dot-com boom/bust",          "mixed"),
        ("2008-07", "2009-06", "financial crisis peak",      "high"),  # base-effect: expectations elevated
        ("2020-01", "2020-12", "COVID shock",                "low"),
    ],
    "dividend_growth": [
        ("1982-07", "1983-06", "post-recession recovery",    "high"),
        ("1990-07", "1991-06", "Gulf War recession",         "low"),
        ("2007-07", "2009-12", "financial crisis",           "low"),
        ("2020-01", "2020-12", "COVID shock",                "low"),
    ],
    "returns_1yr": [
        ("2002-07", "2003-06", "post dot-com trough",        "high"),
        ("2008-07", "2009-06", "financial crisis peak",      "low"),
        ("2020-01", "2020-06", "COVID shock",                "low"),
        ("2020-07", "2021-06", "COVID recovery",             "high"),
    ],
    "returns_10yr": [
        ("2002-07", "2003-06", "post dot-com trough",        "high"),
        ("2008-07", "2009-06", "financial crisis peak",      "high"),  # long-run expectations rose after crash
        ("2020-01", "2020-06", "COVID shock",                "mixed"),
        ("2021-01", "2022-12", "post-COVID inflation",       "mixed"),
    ],
}

for series_name, episodes in VALIDATION_EPISODES.items():
    if series_name not in extrap_results: continue
    df = extrap_results[series_name]

    print(f"\n{'='*60}")
    print(f"  {series_name} — episode validation")
    print(f"{'='*60}")
    print(f"  {'Episode':<35} {'Expected':>10} {'Mean pred':>12} {'Direction':>12}")
    print(f"  {'─'*72}")

    full_mean = df["predicted"].mean()
    full_std  = df["predicted"].std()

    for start, end, label, expected_dir in episodes:
        s  = pd.Timestamp(start)
        e  = pd.Timestamp(end)
        ep = df[(df["wave_date"] >= s) & (df["wave_date"] <= e)]
        if len(ep) == 0:
            print(f"  {label:<35} {'no data':>10}")
            continue

        ep_mean = ep["predicted"].mean()
        z_score = (ep_mean - full_mean) / (full_std + 1e-9)

        if z_score > 0.3:
            direction = "above avg"
        elif z_score < -0.3:
            direction = "below avg"
        else:
            direction = "near avg"

        match = "✓" if (
            (expected_dir == "high"  and z_score > 0.3) or
            (expected_dir == "low"   and z_score < -0.3) or
            (expected_dir == "mixed")
        ) else "✗"

        print(f"  {label:<35} {expected_dir:>10} {ep_mean:>12.4f}  "
              f"{direction:>12}  {match}")

## 7. Coverage and reliability diagnostics

Article coverage per decade and Mahalanobis distance over time.
Sparse decades with few articles per wave will have less reliable predictions.

In [ ]:
n_series = len(extrap_results)
n_cols   = min(2, n_series)
n_rows   = 2 * ((n_series + n_cols - 1) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 5 * n_rows))
fig.suptitle("Extrapolation reliability diagnostics", fontweight="bold")
axes = np.array(axes).reshape(n_rows, n_cols)

for col_idx, (series_name, df) in enumerate(extrap_results.items()):
    c   = col_idx % n_cols
    df  = df.copy()
    df["year"]   = df["wave_date"].dt.year
    df["decade"] = (df["year"] // 10 * 10).astype(str) + "s"

    # Top panel: articles per wave by decade
    ax = axes[(col_idx // n_cols) * 2, c]
    dec_coverage = df.groupby("decade")["n_articles_kept"].median()
    colors_dec   = ["#c0392b" if v < 10 else "#e07b39" if v < 30 else "#4a9e6b"
                    for v in dec_coverage]
    ax.bar(dec_coverage.index, dec_coverage.values, color=colors_dec,
           alpha=0.85, edgecolor="white")
    ax.axhline(10, color="red", ls="--", lw=1, alpha=0.7, label="10-article floor")
    ax.set_xlabel("Decade"); ax.set_ylabel("Median articles/wave")
    ax.set_title(f"{series_name.replace('_',' ').title()}\nCoverage by decade")
    ax.legend(fontsize=7); ax.grid(alpha=0.3, axis="y")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    # Bottom panel: Mahalanobis distance over time
    ax = axes[(col_idx // n_cols) * 2 + 1, c]
    ax.fill_between(df["wave_date"], 0, df["mahal_dist"],
                    alpha=0.4, color="#2c5f8a", label="Mahal. distance")
    thresh = saved_models[series_name]["mahal_thresh"]
    ax.axhline(thresh, color="#c0392b", ls="--", lw=1.5,
               label=f"Threshold ({MAHAL_PERCENTILE:.0%})")
    first_cal = df[df["in_calibration"]]["wave_date"].min()
    ax.axvline(first_cal, color="black", ls="--", lw=1, alpha=0.5)
    for start, end in NBER_RECESSIONS:
        s = pd.Timestamp(start + "-01"); e = pd.Timestamp(end + "-01")
        if e >= df["wave_date"].min():
            ax.axvspan(s, e, alpha=0.08, color="gray")
    ax.set_xlabel("Wave date"); ax.set_ylabel("Mahalanobis distance")
    ax.set_title("Distance from calibration support")
    ax.legend(fontsize=7); ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "extrapolation_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Calibration period fit check

In the calibration period we have both predicted and realized values.
This is the clearest check on whether the model is working correctly.

In [ ]:
n_series = len(extrap_results)
n_cols   = min(2, n_series)
n_rows   = (n_series + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(8 * n_cols, 5 * n_rows), squeeze=False)
fig.suptitle("Calibration period fit", fontweight="bold")

for idx, (series_name, df) in enumerate(extrap_results.items()):
    ax  = axes[idx // n_cols, idx % n_cols]
    cal = df[df["in_calibration"]].dropna(subset=["realized"])
    if len(cal) == 0:
        ax.text(0.5, 0.5, "No calibration overlap", ha="center", va="center")
        continue

    r2 = 1 - np.sum((cal["realized"] - cal["predicted"])**2) /              np.sum((cal["realized"] - cal["realized"].mean())**2)

    ax.scatter(cal["realized"], cal["predicted"], alpha=0.7, color="#2c5f8a", s=30)
    lo = min(cal["realized"].min(), cal["predicted"].min())
    hi = max(cal["realized"].max(), cal["predicted"].max())
    ax.plot([lo,hi],[lo,hi],"k--",lw=1)
    ax.set_xlabel("Realized survey"); ax.set_ylabel("Predicted")
    ax.set_title(f"{series_name.replace('_',' ').title()}  R²={r2:.3f}  n={len(cal)}",
                 fontweight="bold")
    ax.grid(alpha=0.3)
    ax.text(0.05, 0.92, f"R²={r2:.3f}", transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

# Hide unused subplots
for idx in range(n_series, n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "calibration_fit.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n── Extrapolation Summary ─────────────────────────────────────────────────")
for series_name, df in extrap_results.items():
    wf_r2  = all_results[series_name]["wf_r2"]
    alpha  = saved_models[series_name]["alpha"]
    pre    = df[~df["in_calibration"]]
    cal    = df[ df["in_calibration"]]

    print(f"\n{series_name}  [RIDGE  α={alpha:.2e}  WF R²={wf_r2:.3f}]")
    print(f"  Total waves:          {len(df):,}")
    if len(pre):
        print(f"  Pre-survey:           {len(pre):,}  "
              f"({pre['wave_date'].min().date()} → {pre['wave_date'].max().date()})")
    print(f"  Calibration overlap:  {len(cal):,}  "
          f"({cal['wave_date'].min().date()} → {cal['wave_date'].max().date()})")
    print(f"  Out-of-support:       {df['out_of_support'].sum():,}  "
          f"({df['out_of_support'].mean():.1%})")
    print(f"  Median articles/wave: {df['n_articles_kept'].median():.0f}")
    print(f"  Saved:                output/{series_name}/extrapolated_{series_name}.csv")